In [1]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
import time

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from utils import eda_toolkit as toolkit
from utils import build_features as bf
from utils import trainer_toolkit as tt
from utils import test_toolkit as ts

from eval.scorer import DemandScorer
from eval.holdout import build_holdout_dataset, evaluate_on_holdout
from models.baselines import seasonal_naive, historical_median
from models.lgbm_tweedie_wrapper import LightGBMTweedieWrapper
from trainer.walk_forward import WalkForwardTrainer

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

RESULTS_PATH = PROJECT_ROOT / "results" / "lgbm"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

train = pd.read_parquet(DATA_PROCESSED / "train_k54.parquet")
holdout = pd.read_parquet(DATA_PROCESSED / "holdout_k54.parquet")

feature_builder = lambda df: bf.build_features(df, dropna=True)

assert train.index.max() < holdout.index.min(), "Solapamiento temporal train/holdout"

print("=" * 70)
print(f"Train:   {len(train):>6} filas  ({train.index[0]} -> {train.index[-1]})")
print(f"Holdout: {len(holdout):>6} filas  ({holdout.index[0]} -> {holdout.index[-1]})")
print("=" * 70)

Train:    56992 filas  (2023-01-01 04:30:00 -> 2025-05-26 17:15:00)
Holdout:  14248 filas  (2025-05-26 17:30:00 -> 2025-12-31 20:30:00)


# Introducción

El Gradient Boosting con LightGBM es un marco de aprendizaje supervisado que construye un ensamble aditivo de árboles de decisión mediante la optimización secuencial de una función de pérdida diferenciable. A diferencia de los modelos que imponen una ecuación paramétrica para la evolución de la media condicional, LightGBM aprende la relación entre las variables predictoras y la respuesta directamente a partir de los datos, sin asumir una forma funcional específica.

En el contexto de la demanda de pasajeros por franja horaria, esta arquitectura ofrece una alternativa a los modelos recurrentes y a los enfoques econométricos tradicionales, al permitir que la relación entre el calendario, la estacionalidad, los rezagos y la demanda futura sea aprendida directamente de los datos. La naturaleza de la respuesta, sin embargo, impone una restricción sobre la función de pérdida: la serie presenta sobredispersión marcada en las horas de mayor operación y una proporción apreciable de franjas con cero pasajeros, estructura que una pérdida cuadrática o una verosimilitud Poisson subestimarían. Por esta razón se adopta la distribución Tweedie como familia de verosimilitud del objetivo, cuya forma compuesta Poisson-Gamma asigna masa puntual en cero y una densidad continua para los valores positivos.

El objetivo de esta sección es presentar, entrenar y evaluar modelos LightGBM con pérdida Tweedie para la predicción de la demanda de pasajeros por franja horaria en la ruta K54 bajo el protocolo del proyecto. Dado que la operación requiere pronósticos a múltiples escalas temporales, el problema se formula como una tarea de predicción directa multi-horizonte: para cada intervalo operativo solicitado —30, 60 y 120 minutos— se entrena y optimiza un modelo LightGBM independiente, cada uno con su propio objetivo agregado y su propia validación. La evaluación sobre el *holdout* replica las condiciones de despliegue operativo mediante reentrenamiento rodante por bloques con ventana deslizante, respetando la partición cronológica y las condiciones de información establecidas para los demás modelos: uso exclusivo de variables disponibles en el instante de decisión y ausencia de fuga de información entre conjuntos.

### Arquitectura

LightGBM construye un modelo aditivo de árboles de decisión que aproxima la función objetivo de forma secuencial. Para un conjunto de $N$ observaciones $\{\mathbf{x}_i, y_i\}_{i=1}^{N}$, el modelo final se expresa como la suma de $M$ árboles $h_m(\cdot)$ escalados por una tasa de aprendizaje $\nu$:

$$
F_M(\mathbf{x}_i) = \sum_{m=1}^{M} \nu \, h_m(\mathbf{x}_i),
$$

donde cada árbol $h_m \in \mathcal{H}$ es una función escalonada sobre el espacio de predictores. El ajuste es *forward stagewise*: en la iteración $m$, el modelo previo $F_{m-1}$ se mantiene fijo y se añade un nuevo árbol que corrige sus errores,

$$
F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu \, h_m(\mathbf{x}).
$$

El nuevo árbol se obtiene minimizando una expansión de segundo orden de la pérdida $\mathcal{L}$ alrededor de la predicción actual. Definiendo el gradiente $g_{im}$ y el hessiano $h_{im}$ de la pérdida respecto a $F_{m-1}(\mathbf{x}_i)$,

$$
g_{im} = \left. \frac{\partial \mathcal{L}(y_i, F)}{\partial F} \right|_{F = F_{m-1}(\mathbf{x}_i)}, \qquad
h_{im} = \left. \frac{\partial^2 \mathcal{L}(y_i, F)}{\partial F^2} \right|_{F = F_{m-1}(\mathbf{x}_i)},
$$

las hojas del árbol se calculan con la solución de Newton,

$$
w_{j}^{*} = - \frac{\sum_{i \in I_j} g_{im}}{\sum_{i \in I_j} h_{im} + \lambda},
$$

donde $I_j$ es el conjunto de observaciones que caen en la hoja $j$ y $\lambda$ es el término de regularización $\ell_2$. La ganancia de una partición que divide $I$ en $I_L$ e $I_R$ se evalúa como:

$$
\mathcal{G} = \frac{1}{2}\left[
\frac{\left(\sum_{i \in I_L} g_{im}\right)^2}{\sum_{i \in I_L} h_{im} + \lambda} +
\frac{\left(\sum_{i \in I_R} g_{im}\right)^2}{\sum_{i \in I_R} h_{im} + \lambda} -
\frac{\left(\sum_{i \in I} g_{im}\right)^2}{\sum_{i \in I} h_{im} + \lambda}
\right].
$$

Bajo el objetivo `tweedie`, LightGBM asume que la respuesta sigue una distribución Tweedie con parámetro de potencia $p$ y utiliza una función de enlace logarítmica, de modo que la media condicional se obtiene como $\hat{\mu}(\mathbf{x}) = \exp(F(\mathbf{x}))$. La pérdida que se minimiza es la desviación negativa de la log-verosimilitud Tweedie,

$$
\mathcal{L}_{\text{Tweedie}}(y, \hat{\mu}) = - \left[ y \, \frac{\hat{\mu}^{1-p}}{1-p} - \frac{\hat{\mu}^{2-p}}{2-p} \right].
$$

:::{admonition} Información: Distribución Tweedie
:class: info

La distribución Tweedie con parámetro de potencia $p$ pertenece a la familia exponencial y se caracteriza por la relación entre la media $\mu$ y la varianza:

$$
\operatorname{Var}(y) = \phi \, \mu^{p}, \qquad 1 < p < 2.
$$

Valores de $p$ cercanos a 1 aproximan la verosimilitud a una Poisson; valores cercanos a 2 la aproximan a una Gamma; los valores intermedios corresponden a la familia compuesta Poisson-Gamma, que admite masa en cero y una componente continua positiva. En este proyecto, el parámetro $p$ se estimó empíricamente a partir de la relación varianza-media de la serie, obteniendo un valor de $p = 1.28$, el cual se fijó como constante durante el entrenamiento para garantizar la estabilidad y comparabilidad de la métrica.
:::

Con la función de enlace logarítmica, las derivadas de la pérdida respecto a la predicción del modelo son:

$$
\frac{\partial \mathcal{L}}{\partial F} = -y \, e^{(1-p)F} + e^{(2-p)F}, \qquad
\frac{\partial^2 \mathcal{L}}{\partial F^2} = -y(1-p) \, e^{(1-p)F} + (2-p) \, e^{(2-p)F}.
$$

Estas expresiones alimentan el gradiente $g_{im}$ y el hessiano $h_{im}$ en cada iteración del boosting y son las que permiten que el modelo pondere de forma diferenciada las franjas con cero pasajeros y las franjas de alta demanda. Para $1 < p < 2$, el hessiano es estrictamente positivo, lo que garantiza la convexidad local del subproblema de optimización en cada hoja.

El modelo completo $F_M(\cdot)$ asigna a cada vector de predictores $\mathbf{x}$ (hora, día de la semana, mes, codificación cíclica, estacionalidad, rezagos y tendencia temporal) una predicción de la media condicional de pasajeros acumulados en la ventana objetivo, $\hat{\mu}(\mathbf{x}) = \exp(F_M(\mathbf{x}))$.

### Protocolo de entrenamiento y validación

El problema se formula como un pronóstico supervisado directo multi-horizonte: para atender los tres intervalos operativos requeridos sobre la ruta (30, 60 y 120 minutos, equivalentes a $h \in \{2, 4, 8\}$ franjas de 15 minutos), se entrena un modelo LightGBM independiente por cada horizonte. Cada modelo se ajusta sobre su propio *target*, construido como la suma rodante de pasajeros desde el instante de decisión hasta el final del intervalo, compartiendo el mismo conjunto de **23 variables predictoras** descrito en la fase de ingeniería de variables (incluyendo la variable de tendencia `t_index` y los rezagos operativos $\ge 32$ franjas). La separación en tres modelos respeta la formulación directa: cada intervalo se predice en un único paso, sin emplear la predicción de un horizonte como insumo de otro.

Para mitigar el efecto del cambio de régimen estructural observado entre 2023 y 2025 (el cual provoca que modelos entrenados con todo el histórico sobreestimen la demanda futura), se implementó una estrategia de **ventana deslizante** (*sliding window*). Durante la validación cruzada, el hiperparámetro `window` define cuántas franjas de historia reciente observa el modelo en cada pliegue, explorando tamaños entre 18,000 y 26,000 franjas (aproximadamente entre 8 y 12 meses de operación) frente a un tramo inicial de calentamiento fijado en 26,000 franjas. Esta acotación temporal equilibra la captura del ciclo estacional anual con la adaptación al nivel de demanda más reciente, superando al esquema de expansión infinita (*expanding window*).

La validación sobre `train.parquet` se realiza mediante un esquema *walk-forward* de 5 pliegues consecutivos. Cada pliegue entrena con el bloque de historia delimitado por `window` y valida sobre el bloque cronológico inmediatamente posterior, aplicando un *embargo* igual al horizonte de predicción menos uno ($h-1$) para evitar la fuga de información en la frontera. Una vez seleccionados los hiperparámetros óptimos, la evaluación sobre `holdout.parquet` unifica exactamente esta misma mecánica: avanza cronológicamente por bloques, reentrenando el modelo en cada corte con las últimas `window` franjas disponibles hasta la frontera de embargo.

La implementación se realiza utilizando la librería `LightGBM` en su versión `4.x`, aprovechando su soporte nativo para el objetivo `tweedie`, su rutina de *early stopping* interno (evaluado sobre una partición cronológica del 70/30 dentro de cada bloque de entrenamiento, separada también por el embargo del horizonte) y su capacidad para manejar valores faltantes sin preprocesamiento adicional.

## Ventana de predicción de 30 minutos

### Entrenamiento

Para la selección de hiperparámetros del modelo LightGBM correspondiente al horizonte de 30 minutos ($h = 2$ franjas), se implementó una validación cruzada temporal de 5 pliegues sobre el conjunto de entrenamiento. La búsqueda de Optuna (100 ensayos) exploró la complejidad de los árboles de decisión, los parámetros de regularización y el tamaño de la ventana de historia reciente.

El modelo recibe directamente una matriz tabular con las 23 variables predictoras (calendario, codificación cíclica, estacionalidad, tendencia temporal y rezagos precalculados). Para evitar el sesgo de anticipación (*look-ahead bias*) y adaptarse al cambio de régimen estructural de la demanda observado en la serie, se implementó una estrategia de ventana deslizante (*sliding window*). En cada pliegue, el modelo se entrena con un bloque de historia delimitado por el hiperparámetro `window` y se valida con el bloque cronológico inmediatamente posterior, aplicando un *embargo* de una franja ($h - 1 = 1$) para garantizar la integridad temporal de los datos.

El espacio de búsqueda incluyó el tamaño de la ventana de entrenamiento (`window`), el número máximo de hojas por árbol (`num_leaves`), la tasa de aprendizaje (`learning_rate`), el mínimo de datos y de suma de hessianos por hoja (`min_data_in_leaf`, `min_sum_hessian_in_leaf`), las fracciones y frecuencia de submuestreo (`bagging_fraction`, `bagging_freq`, `feature_fraction`) y las penalizaciones por regularización $\ell_1$ y $\ell_2$. El entrenamiento se configuró con un máximo de 5,000 árboles y una paciencia de 300 iteraciones para la parada temprana (*early stopping*) interna, evaluada sobre el último 30% cronológico de cada bloque de entrenamiento (respetando el embargo de una franja). El desempeño se evaluó mediante la desviación Tweedie ($p = 1.28$, con un límite superior de 100 por observación para mitigar eventos atípicos) sobre los bloques de validación.

In [ ]:
WINDOW_MAX = 26000

def lgbm_search_space(trial: optuna.Trial) -> dict:
    return {
        "window": trial.suggest_categorical(
            "window", [18000, 20000, 22000, 24000, WINDOW_MAX]
        ),
        "num_leaves": trial.suggest_int("num_leaves", 15, 300, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 500, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0, step=0.05),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0, step=0.05),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 50.0, log=True),
        "min_sum_hessian_in_leaf": trial.suggest_float(
            "min_sum_hessian_in_leaf", 1e-3, 10.0, log=True
        ),
    }

exec_id = 6
exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=2,
    n_splits=5,
    warmup=WINDOW_MAX,
    embargo=1,
    n_trials=100,
    suggest_params_fn=lgbm_search_space,
    early_stop_rounds=30,
    inner_val_frac=0.3,
    model_kwargs={
        "tweedie_variance_power": 1.28,
        "n_estimators": 5000,
        "early_stopping_rounds": 300,
        "num_threads": 8,
        "seed": 42,
    },
)

study = trainer.run(train)
best_params = dict(study.best_params)

  0%|          | 0/100 [00:00<?, ?it/s]

Respecto a la eficiencia del algoritmo de búsqueda, la optimización completó la totalidad de las 100 evaluaciones programadas sin que se activara el mecanismo de parada temprana (*early stopping* de 30 ensayos consecutivos sin mejora), lo que refleja una exploración continua de mejoras a lo largo de toda la trayectoria de búsqueda. El mecanismo de poda (*pruning*) no descartó ninguna configuración (0 ensayos podados y 0 fallidos), confirmando que el modelo completó los 5 pliegues de validación cruzada en la totalidad de los casos. La mejor configuración alcanzó una Desviación Tweedie de 5.9775.

El entrenamiento efectivo de los árboles a través de los 500 pliegues evaluados (100 ensayos $\times$ 5 pliegues) acumuló un tiempo neto de 84.20 minutos (5,052.25 segundos), equivalente a un promedio de 10.10 segundos por pliegue bajo el esquema de ventana deslizante (*sliding window*). La ejecución completa tuvo una duración de 1 hora, 42 minutos y 46 segundos (6,166 segundos); la diferencia de 18.56 minutos (1,113.75 segundos) respecto al tiempo acumulado de los pliegues responde al ajuste inicial sobre el bloque de calentamiento en cada ensayo y a la evaluación de las métricas de validación en memoria. Dado que LightGBM realiza la inferencia de forma vectorizada, la generación de pronósticos sobre el conjunto de prueba (*holdout*) se ejecuta en fracciones de segundo en cada bloque rodante.

In [3]:
resumen_data = []

exec_id = 6
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_tweedie = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(
            study_name="LightGBMTweedieWrapper_h2_study", 
            storage=storage_url
        )

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])
        
        mejor_tweedie = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor Tweedie Dev": mejor_tweedie,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Fallidos,Mejor Tweedie Dev,Tiempo Total (s),Tiempo por Fold (s)
100,100,0,0,5.9775,5052.2500,10.1040


El ranking de configuraciones mostró que las cinco mejores opciones presentaron una Desviación Tweedie en torno a 5.98. Dado que la variabilidad entre los pliegues de validación se mantuvo relativamente alta ($\sigma \approx 1.99$), las diferencias de milésimas entre estas configuraciones representan un empate estadístico. Sin embargo, el proceso mostró una notable estabilidad en la selección de hiperparámetros: los cinco mejores modelos convergieron hacia la ventana de historia de 24,000 franjas (descartando el límite superior de 26,000), un submuestreo pronunciado de predictores, fracciones elevadas de muestreo de filas, tasas de aprendizaje bajas y penalizaciones $\ell_1$ y $\ell_2$ cercanas a cero.

La selección consistente de 24,000 franjas (equivalentes a poco más de un año calendario, dado que cada franja representa 15 minutos) permite capturar el ciclo estacional anual completo evitando incorporar la historia más antigua de 2023, la cual presentaba un régimen de demanda más elevado que inducía sesgos de sobrepredicción sobre los datos recientes. Asimismo, la combinación de tasas de aprendizaje bajas con un submuestreo estricto de variables y árboles de complejidad intermedia resultó suficiente para regularizar el ensamble y mitigar el sobreajuste sin requerir penalizaciones explícitas sobre los pesos de las hojas.

In [5]:
exec_dir = RESULTS_PATH / "exec_6"
df_lgbm = pd.read_csv(exec_dir / "LightGBMTweedieWrapper_results.csv")

df_lgbm["lr"] = df_lgbm["learning_rate"].apply(lambda x: f"{x:.4f}")
df_lgbm["l1"] = df_lgbm["lambda_l1"].apply(lambda x: f"{x:.1e}")
df_lgbm["l2"] = df_lgbm["lambda_l2"].apply(lambda x: f"{x:.1e}")

cols_agrupar = [
    "window", 
    "num_leaves", 
    "min_data_in_leaf", 
    "feature_fraction", 
    "bagging_fraction", 
    "lr", 
    "l1",
    "l2"
]

df_agg = df_lgbm.groupby(cols_agrupar)["tweedie_deviance"].agg(["mean", "std"]).reset_index()

df_agg["Tweedie Deviance"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)

df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "window": "Ventana",
    "num_leaves": "Hojas",
    "min_data_in_leaf": "Min Data Hoja",
    "feature_fraction": "Feat. Frac",
    "bagging_fraction": "Bagging Frac",
    "lr": "LR",
    "l1": "L1",
    "l2": "L2"
})

df_final = df_top5[
    ["Ventana", "Hojas", "Min Data Hoja", "Feat. Frac", "Bagging Frac", "LR", "L1", "L2", "Tweedie Deviance"]
]

tt.display_styled_df(df=df_final, decimals=4, align_left_first=False)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"24,000",63,11,0.4000,0.9500,0.0019,3.4e-08,3.0e-08,5.978 ± 1.986
"24,000",61,10,0.4000,0.9500,0.0011,1.9e-06,3.3e-08,5.978 ± 1.991
"24,000",90,11,0.4000,1.0000,0.0033,5.5e-07,1.5e-08,5.979 ± 2.002
"24,000",67,11,0.4000,1.0000,0.0025,8.1e-08,1.0e-08,5.981 ± 1.996
"24,000",76,11,0.4000,0.9500,0.0013,6.1e-08,1.8e-07,5.982 ± 1.972


La selección del modelo ganador entre las cinco mejores configuraciones se resolvió a favor de la primera (Desviación Tweedie de 5.978). Aunque registra la misma media que la segunda opción, el primer modelo ofrece un balance entre desempeño y complejidad: presenta una ligera reducción en la variabilidad entre pliegues ($\sigma = 1.986$ frente a $\sigma = 1.991$), emplea una tasa de aprendizaje marginalmente superior (0.0019 frente a 0.0011) que favorece la convergencia sin sacrificar generalización y mantiene una complejidad similar (~63 hojas) apoyada por el submuestreo de predictores.

In [2]:
best_params = {
    "window": 24000,
    "num_leaves": 63,
    "learning_rate": 0.001915947545430018,
    "min_data_in_leaf": 11,
    "feature_fraction": 0.4,
    "bagging_fraction": 0.9500000000000001,
    "bagging_freq": 9,
    "lambda_l1": 3.4040683712300666e-08,
    "lambda_l2": 3.010732420824225e-08,
    "min_sum_hessian_in_leaf": 0.4049280114035653,
}

window = best_params["window"]
model_params = {k: v for k, v in best_params.items() if k != "window"}

final_kwargs = {
    "tweedie_variance_power": 1.28,
    "n_estimators": 5000,
    "early_stopping_rounds": 300,
    "inner_val_frac": 0.3,
    "num_threads": 8,
    "seed": 42,
}

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=RESULTS_PATH / "exec_6",
    feature_builder=feature_builder,
    horizon_slots=2,
    n_splits=5,
    warmup=26000,
    embargo=1,
    n_trials=100,
    model_kwargs={}
)

X_train_full, y_train_full, _, _ = trainer._build_dataset(train)

model_production = LightGBMTweedieWrapper(**model_params, **final_kwargs)
model_production.fit(X_train_full.iloc[-window:], y_train_full[-window:])

Para evaluar la estructura de los residuos tanto en el conjunto de entrenamiento (*In-Sample*) como en el de prueba (*Out-of-Sample*), a partir de las demandas predichas y las observaciones reales, se calcularon los residuos y su versión estandarizada mediante las siguientes expresiones:

$$
e_t = y_t - \hat{y}_t, \qquad z_t = \frac{e_t - \bar{e}}{s_e}
$$

donde $y_t$ es la demanda observada, $\hat{y}_t$ es la predicción del modelo, $\bar{e}$ es la media muestral de los residuos y $s_e$ es su desviación estándar.

Debido a que los estadísticos de Ljung-Box y ARCH-LM escalan linealmente con el tamaño de la muestra, evaluar la totalidad de la ventana activa de entrenamiento ($24000$ franjas) introduce un efecto de sobre-potencia (*over-power*) que infla los estadísticos ante cualquier dependencia residual. Para atenuar este sesgo sin alterar la secuencia cronológica de los rezagos, el diagnóstico se realizó sobre el último bloque continuo de entrenamiento ($N = 1000$ franjas). A pesar de esta reducción muestral, las pruebas de Ljung-Box (autocorrelación) y ARCH-LM (heterocedasticidad) rechazaron sus hipótesis nulas en todos los rezagos evaluados ($p < .001$).

In [3]:
preds_train_final = model_production.predict_in_sample()
residuals_train = y_train_full[-window:] - preds_train_final

block_size = 1000
residuals_block = residuals_train[-block_size:]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_train = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_train,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),215.137,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),232.813,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),256.750,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),81.281,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),82.514,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),94.387,< .001,< .001,Efectos ARCH Significativos


La persistencia de autocorrelación residual obedece a la exclusión operativa de los rezagos intradía recientes ($k < 32$) y a la superposición del horizonte de predicción ($h = 2$), factores que trasladan los choques de corto plazo al término de error. Por su parte, la detección de efectos ARCH refleja la relación media-varianza propia de la demanda de pasajeros $\left(\operatorname{Var}(y) = \phi\,\mu^{1.28}\right)$, donde la dispersión aumenta en las franjas de alta afluencia. Este comportamiento es característico de los conteos de transporte con sobredispersión y la función de pérdida Tweedie logró asimilarlo sin comprometer la capacidad predictiva de la media condicional.

La inspección visual evidenció que el modelo LightGBM-Tweedie reprodujo tanto los ciclos estacionales regulares de la operación de SOBUSA, las mesetas de demanda en días hábiles frente a los valles de fin de semana, como las contracciones atípicas registradas entre 2024 y 2025, incluyendo las caídas de cierre de año, Semana Santa y la interrupción operativa de septiembre de 2024.

In [11]:
ts.plot_temporal_fit_panel(
    dates=X_train_full.index[-window:],
    y_true=y_train_full[-window:],
    y_pred=preds_train_final,
    zoom_left=("2024-09-02", "2024-09-08"),
    zoom_right=("2025-02-12", "2025-02-12"),
    text_left="Septiembre 2024",
    text_right="12 Febrero 2025",
    title="Demanda de pasajeros 2024-2025",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros por día",
    ylab_zoom="Pasajeros por franja",
    zoom_ylim_pad=(0.10, 0.12),
    hspace=0.1, wspace = 0.05
)

En el detalle intradía del 2 al 8 de septiembre de 2024 y del 12 de febrero de 2025, la predicción presentó una ligera subestimación en los picos matutinos máximos (alcanzando aproximadamente $1100$ pasajeros por franja frente a máximos reales cercanos a $1400$ el 5 de septiembre, y alrededor de $950$ frente a $1050$ el 12 de febrero), comportamiento propio de un estimador de media condicional sujeto a la restricción operativa de rezagos ($k < 32$). A pesar de este suavizamiento en las crestas de alta frecuencia, la estructura basada en árboles de decisión logró capturar con precisión la transición de hora pico matutina, la meseta de media mañana y el segundo repunte vespertino a lo largo de la serie.

### Evaluación

La evaluación del rendimiento del modelo LightGBM-Tweedie requiere contrastar su capacidad de ajuste sobre los datos observados con su capacidad predictiva sobre demanda no vista. Mientras que la ventana activa de entrenamiento (*In-Sample*, $w = 24000$ franjas) permite verificar la convergencia del algoritmo a la dinámica histórica de operación, la evaluación fuera de muestra (*Out-of-Sample*, conjunto *holdout* del `2025-05-26` al `2025-12-31`, $N = 14{,}247$ franjas) determina la utilidad operativa real del modelo ante nuevos regímenes de movilidad.

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se aplicaron los criterios establecidos en el protocolo experimental. Estos criterios se basan en el error relativo porcentual entre las métricas de prueba y entrenamiento, denotado como:

$$
\Delta_{\text{rel}} (\%) = \frac{M_{\text{test}} - M_{\text{train}}}{|M_{\text{train}}|} \times 100
$$

Se estableció un umbral de tolerancia relativo de $|\Delta_{\text{rel}}| = 5\%$ sobre la función de pérdida principal (*Tweedie Deviance*, con parámetro de potencia $p = 1.28$). Un incremento relativo superior a este margen evidenciaría degradación en la capacidad de generalización.

No obstante, en el contexto del pronóstico de demanda de pasajeros bajo verosimilitud Tweedie, debe considerarse la naturaleza asimétrica de esta función de pérdida y su sensibilidad diferenciada entre franjas de baja afluencia (incluyendo cierres operativos en cero) y horas pico. Dado que un desplazamiento puntual en la demanda real afecta de forma no lineal las métricas agregadas, la lectura de $\Delta_{\text{rel}}$ se complementó con métricas de escala (`MAE`, `RMSE`), métricas relativas (`SMAPE`, `MASE`), el diagnóstico de sesgo (`Bias`) y la inspección visual de las trayectorias diarias e intradía. En particular, un sesgo positivo en prueba no indica sobreajuste estructural de los árboles, sino la incidencia de caídas atípicas por días festivos y cierres de fin de año frente a la envolvente semanal regular.

Las predicciones fuera de muestra se obtuvieron ejecutando el método `forecast` de `LightGBMTweedieWrapper` sobre la serie continua concatenada (garantizando la disponibilidad exacta de los rezagos operativos sin pérdida de observaciones iniciales ni fuga de información). Este procedimiento aplicó un esquema de reentrenamiento periódico por bloques (`n_splits = 5`) con ventana deslizante fija (`window = 24000`, `embargo = 1`), generando pronósticos puntuales en la escala original de pasajeros por franja de 15 minutos.

In [22]:
df_eval_context = pd.concat([train, holdout], axis=0)
X_eval_all, y_eval_all, y_naive_eval_all, _ = trainer._build_dataset(df_eval_context)

holdout_mask = X_eval_all.index >= holdout.index.min()
X_holdout_final = X_eval_all.loc[holdout_mask]
y_holdout_final = y_eval_all[holdout_mask]
y_naive_holdout_final = y_naive_eval_all[holdout_mask]

start_time = time.time()
preds_holdout = model_production.forecast(
    X_seq=X_holdout_final,
    y_seq=y_holdout_final,
    window=window,
    embargo=trainer.embargo,
    n_splits=trainer.n_splits,
)
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": X_holdout_final.index,
    "y_true": y_holdout_final,
    "y_naive": y_naive_holdout_final,
    "y_pred": preds_holdout,
})

output_path = RESULTS_PATH / "lgbm_holdout_predictions.csv"
df_preds.to_csv(output_path, index=False)

El análisis de los residuos fuera de muestra ($e_t = y_t - \hat{y}_t$, evaluado sobre las últimas $1000$ franjas del conjunto *holdout*) evidenció persistencia de estructura temporal y heterocedasticidad condicional en el error de pronóstico. Las pruebas de Ljung-Box rechazaron la hipótesis nula de independencia serial en todos los rezagos evaluados $\left(p_\text{adj} < .001\right)$, confirmando la presencia de autocorrelación remanente de corto y mediano plazo. De forma paralela, la prueba ARCH-LM detectó efectos ARCH altamente significativos en los tres horizontes evaluados $\left(p_\text{adj} < .001\right)$. Esto indica que, si bien el modelo *LightGBM-Tweedie* captura con precisión la envolvente diaria e intradía de la demanda esperada, persisten fluctuaciones de alta frecuencia de 15 minutos y agrupamiento de varianza residual asociados a la variabilidad operativa propia de las franjas pico y los cierres atípicos de fin de año.

In [6]:
residuals_test = (
    df_preds["y_true"].to_numpy() - df_preds["y_pred"].to_numpy()
)

block_size = 1000
residuals_block = residuals_test[-block_size:]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_test = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_test,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),1244.180,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),1424.257,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),1623.666,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),344.957,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),353.584,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),363.681,< .001,< .001,Efectos ARCH Significativos


De acuerdo con los criterios establecidos en el protocolo experimental, no existe evidencia de sobreajuste (*overfitting*) ni de subajuste (*underfitting*) en el modelo LightGBM-Tweedie. La función de pérdida principal (Tweedie Deviance) mantiene su incremento relativo dentro del umbral de tolerancia pre-establecido, lo que apoya una adecuada capacidad de generalización fuera de muestra. Este comportamiento es consistente con la estabilidad observada en las métricas de error absoluto, mediano y relativo (`MAE`, `Pinball` y `SMAPE`), donde las discrepancias entre entrenamiento y prueba resultan marginales e incluso favorables en términos porcentuales simétricos.

Las mayores variaciones registradas en las métricas cuadráticas (`RMSE`, `MSE`), el `MASE` y el desplazamiento positivo del `Bias` no responden a un deterioro estructural del modelo, sino a la sensibilidad direccional y cuadrática frente a caídas abruptas y puntuales de demanda. En escenarios atípicos como vísperas de cierre de fin de año y días puente, la inercia de los rezagos semanales y las categorías agregadas de calendario sobrestiman transitoriamente el flujo esperado sin comprometer la precisión del régimen operativo regular.

In [5]:
df_preds = pd.read_csv(RESULTS_PATH / "lgbm_holdout_predictions.csv")

X_train_full, y_train_full, y_naive_train_full, _ = trainer._build_dataset(train)
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])

ts.display_train_test_comparison(
    y_train=y_train_full[-window:],
    preds_train=preds_train_final,
    y_naive_train=y_naive_train_full[-window:],
    y_test=df_preds["y_true"].to_numpy(),
    preds_test=df_preds["y_pred"].to_numpy(),
    y_naive_test=df_preds["y_naive"].to_numpy(),
    p=1.28, error_relativo=True
)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,4.3830,4.5807,+4.5118%
MAE,57.3539,59.7743,+4.2201%
RMSE,79.0286,83.4561,+5.6024%
MSE,6245.5212,6964.9228,+11.5187%
Bias,-1.1963,15.9114,+1430.0946%
SMAPE,0.2282,0.2271,-0.5137%
MASE,0.5832,0.6966,+19.4381%
Pinball (τ=0.50),28.6769,29.8871,+4.2201%


La inspección visual de las trayectorias predichas frente a la demanda realizada en el conjunto *holdout* (Figura X) muestra que el modelo LightGBM-Tweedie reproduce con alta fidelidad la envolvente diaria, la estacionalidad semanal y el perfil multimodal intradía a lo largo del segundo semestre. No obstante, se identifica una ligera tendencia a subestimar la demanda en los picos extremos de afluencia. A escala agregada diaria (panel superior), este fenómeno se manifiesta entre mediados de octubre y noviembre, donde los máximos laborales reales superan levemente la envolvente estimada. A escala de franjas de 15 minutos (paneles inferiores), dicha subestimación se concentra en la cúspide de la hora pico matutina (alrededor de las 05:00–05:30) y en los repuntes abruptos de mediodía, lo cual responde al efecto de suavizado propio de la estimación de la media condicional mediante árboles de decisión frente a fluctuaciones de alta frecuencia. En contraste, durante los días puente de mitad de año (junio–julio) y los cierres operativos de final de diciembre se observa el comportamiento inverso, con episodios puntuales de sobreestimación asociados a caídas atípicas del flujo de pasajeros.

In [31]:
ts.plot_temporal_fit_panel(
    dates=pd.to_datetime(df_preds["date"]),
    y_true=df_preds["y_true"].to_numpy(),
    y_pred=df_preds["y_pred"].to_numpy(),
    zoom_left=("2025-09-01", "2025-09-07"),
    zoom_right=("2025-10-15", "2025-10-15"),
    text_left="Septiembre 2025",
    text_right="15 Octubre 2025",
    title="Demanda de pasajeros Holdout 2025 (Out-of-Sample)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros por día",
    ylab_zoom="Pasajeros por franja",
    zoom_ylim_pad=(0.10, 0.12),
    hspace=0.1,
    wspace=0.05,
)